# Week 2: Labeled Intent Dataset Build

This notebook contains the Week-2 labeled dataset and the verification pipeline. The goal is to build a high-quality, balanced, leakage-free dataset covering the six OXODIN intents:
- `create_task`
- `place_call`
- `answer_question`
- `save_memory`
- `set_timer`
- `out_of_scope`

In [ ]:
# =====================================================================
# IMPORTANT CAVEAT:
# Never strip Swiss-German ß to ss on classifier input.
# That normalisation is reserved for generated output only.
# =====================================================================

# Python List of Rows with Inline German Glosses
dataset_rows = [
    # create_task
    ("remind me to buy milk tonight", "create_task"),
    ("remind me to bye milk", "create_task"), # deliberate typo: bye -> buy
    ("add an item to my to-do list: clean the kitchen", "create_task"),
    ("schedule a task to call the landlord tomorrow", "create_task"),
    ("don't let me forget to lock the garage", "create_task"),
    ("put wash the car on my chores list", "create_task"),
    ("Erinnere mich daran, Milch zu kaufen", "create_task"), # = remind me to buy milk

    # place_call
    ("call Sarah right now", "place_call"),
    ("give my dad a ring", "place_call"),
    ("phone the pizza restaurant", "place_call"),
    ("dial the manager's office", "place_call"),
    ("get David on the line", "place_call"),
    ("please dial my grandmother", "place_call"),
    ("Ruf Mama an", "place_call"), # = call mom

    # answer_question
    ("what is the speed of light", "answer_question"),
    ("how many countries are in Europe", "answer_question"),
    ("who wrote the Harry Potter books", "answer_question"),
    ("why is the sky blue during the day", "answer_question"),
    ("explain how photosynthesis works", "answer_question"),
    ("what is the capital city of France", "answer_question"),
    ("tell me the definition of serendipity", "answer_question"),

    # save_memory
    ("remember that my car keys are on the table", "save_memory"),
    ("keep in mind my Wi-Fi password is password123", "save_memory"),
    ("store this info that my blood type is O positive", "save_memory"),
    ("save the memory that I met John today", "save_memory"),
    ("make a mental note that the garage code is 4321", "save_memory"),
    ("write down that my sister's birthday is June 5th", "save_memory"),
    ("rember that my locker number is 105", "save_memory"), # deliberate typo: rember -> remember

    # set_timer
    ("set a timer for 10 minutes", "set_timer"),
    ("start a stopwatch for cooking eggs", "set_timer"),
    ("wake me up in twenty minutes", "set_timer"),
    ("begin a countdown of 5 minutes", "set_timer"),
    ("timer for thirty seconds please", "set_timer"),
    ("pause the current timer", "set_timer"),
    ("Stelle einen Timer auf 5 Minuten", "set_timer"), # = set a timer for 5 minutes

    # out_of_scope
    ("play some jazz music", "out_of_scope"),
    ("turn off the living room lights", "out_of_scope"),
    ("how long will it take to get to work", "out_of_scope"),
    ("find a coffee shop nearby", "out_of_scope"),
    ("can you tell me a funny joke", "out_of_scope"),
    ("what is my calendar looking like tomorrow", "out_of_scope"),
    ("recommend a good movie to watch tonight", "out_of_scope")
]

In [ ]:
import csv
import os

def build_dataset(csv_path="intents.csv"):
    """
    Loads and validates the dataset from intents.csv.
    Checks for duplicates (leakage) and class balance.
    """
    if not os.path.exists(csv_path):
        # Write intents.csv if it doesn't exist locally from dataset_rows
        print(f"{csv_path} not found, generating it from dataset_rows...")
        with open(csv_path, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["text", "label"])
            for r in dataset_rows:
                writer.writerow(r)
                
    rows = []
    seen_texts = set()
    duplicates = []
    class_counts = {}
    
    with open(csv_path, mode='r', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader, None) # Skip header
        
        for line_num, row in enumerate(reader, start=2):
            if not row or len(row) < 2:
                continue
            text, label = row[0].strip(), row[1].strip()
            
            if text in seen_texts:
                duplicates.append((text, line_num))
            seen_texts.add(text)
            
            class_counts[label] = class_counts.get(label, 0) + 1
            rows.append((text, label))
            
    print("=== DATASET BUILD REPORT ===")
    print(f"Total Rows: {len(rows)}")
    print(f"Unique Sentences: {len(seen_texts)}")
    
    if duplicates:
        print("\nWARNING: Leakage detected! Duplicate entries found:")
        for dup, line in duplicates:
            print(f" - '{dup}' (Line {line})")
    else:
        print("\nClean Check: Dataset is leakage-free (0 duplicates).")
        
    print("\nClass Distribution:")
    for label, count in sorted(class_counts.items()):
        print(f" - {label:18} : {count} rows")
        
    min_count = min(class_counts.values()) if class_counts else 0
    max_count = max(class_counts.values()) if class_counts else 0
    
    if min_count == max_count and len(class_counts) == 6:
        print("\nBalance Check: PERFECTLY BALANCED across all 6 intents.")
    elif max_count - min_count <= 1:
        print("\nBalance Check: Roughly balanced (difference <= 1).")
    else:
        print("\nWARNING: Dataset is imbalanced!")
        
    return rows, class_counts

# Run the build_dataset() function
rows, counts = build_dataset("intents.csv")